# Git 上传（仅 <=100MB 文件）命令清单

本 notebook 记录一套可直接执行的命令：
- 在当前目录创建干净提交历史
- 自动排除大于 100MB 的文件
- 推送到 GitHub 远程仓库（示例远程名：`Backup`）


In [ ]:
cd /data/home/public/qiuqizhi

# 1) 保险备份当前分支
git branch backup-before-size-filter

# 2) 新建无历史分支
git checkout --orphan clean-upload

# 3) 清空索引（保留本地文件）
git rm -r --cached . 2>/dev/null || true

# 4) 先全部加入索引
git add .

# 5) 仅保留 <=100MB 文件（从索引移除大文件，不删除本地）
THRESHOLD_MB=100
THRESHOLD_BYTES=$((THRESHOLD_MB*1024*1024))

git ls-files -z | while IFS= read -r -d '' f; do
  [ -f "$f" ] || continue
  size=$(wc -c < "$f")
  if [ "$size" -gt "$THRESHOLD_BYTES" ]; then
    git rm --cached -- "$f" >/dev/null
    echo "skip(>${THRESHOLD_MB}MB): $f"
  fi
done

# 6) 提交
git commit -m "upload files <= ${THRESHOLD_MB}MB only"

# 7) 切换主分支名为 main
git branch -M main

# 8) 推送到远程（首次）
git push -u Backup main

# 如果提示 non-fast-forward 再执行：
# git push -u Backup main --force


## 说明

- 这套命令会重建本地提交历史（`--orphan`），避免旧历史中的大文件继续被推送。
- `git rm --cached` 只取消跟踪，不删除本地文件。
- 如果远程仓库名不是 `Backup`，先用 `git remote -v` 查看并替换命令中的远程名。

## 日常增量更新（后续每次修改后上传）

如果你已经完成首次上传，后续同步到 GitHub 用下面命令即可：

In [ ]:
cd /data/home/public/qiuqizhi

# 1) 查看改动

git status

# 2) 暂存改动（按需可改为只 add 部分路径）

git add -A

# 3) 提交

git commit -m "update: your change summary"

# 4) 推送到当前跟踪远程分支（你这里是 Backup/main）

git push

# 可选：推送前检查暂存的大文件
# git diff --cached --name-only | xargs -I{} du -h "{}" 2>/dev/null | sort -hr | head
# 如不想上传某文件：
# git restore --staged <path/to/file>